# Phase 1: Baseline RAG in Google Colab

This notebook runs a fully local, open-source RAG pipeline: SentenceTransformers embeddings → normalized FAISS inner-product retrieval → Qwen grounded generation. The reusable implementation lives in `src/rag/baseline.py`; this notebook only orchestrates it.

Before running, replace `REPOSITORY_URL` in the next cell with the URL of your fork/repository.

## Environment

Clone the repository into the Colab runtime and install its free, open-source dependencies.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/YOUR_GITHUB_USERNAME/adaptive-rag.git'
PROJECT_DIR = Path('/content/adaptive-rag')

if not (PROJECT_DIR / 'src').exists():
    if 'YOUR_GITHUB_USERNAME' in REPOSITORY_URL:
        raise RuntimeError('Set REPOSITORY_URL to your GitHub repository URL, then run this cell again.')
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(f'Working directory: {Path.cwd()}')

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU: unavailable — enable a GPU under Runtime > Change runtime type for practical Qwen inference.')

## Build or load the retrieval index

Document vectors are L2-normalized before they are added to `faiss.IndexFlatIP`, so the returned inner-product score is cosine similarity. The saved metadata preserves the exact FAISS-position-to-document mapping.

In [ ]:
from src.rag import FAISSRetriever, RAGConfig, load_documents

config = RAGConfig(
    embedding_model_name='sentence-transformers/all-MiniLM-L6-v2',
    generation_model_name='Qwen/Qwen2.5-1.5B-Instruct',
    top_k=3,
    max_new_tokens=220,
)
corpus_path = Path('data/phase1_corpus.json')
index_dir = Path('data/phase1_faiss_index')
force_rebuild = False

retriever = FAISSRetriever(
    embedding_model_name=config.embedding_model_name,
    device=config.device,
    batch_size=config.embedding_batch_size,
)
if force_rebuild or not (index_dir / 'documents.faiss').exists():
    documents = load_documents(corpus_path)
    retriever.build(documents)
    retriever.save(index_dir)
    print(f'Built and saved an index for {len(documents)} documents.')
else:
    retriever.load(index_dir)
    print(f'Loaded an index for {len(retriever.documents)} documents.')

print('Embedding device:', retriever.device)

## Retrieval

Inspect each ranked passage and its cosine-similarity score before generation.

In [ ]:
def display_retrieval(question: str, top_k: int = 3):
    results = retriever.retrieve(question, top_k=top_k)
    print('Question:', question)
    for rank, result in enumerate(results, start=1):
        print(f'\nRank {rank}')
        print('Title:', result.title)
        print(f'Similarity: {result.score:.4f}')
        print('Text:', result.text)
    return results

sample_question = 'Where is the Mona Lisa displayed?'
sample_results = display_retrieval(sample_question)

## Generation

Qwen runs directly in the Colab runtime. The prompt requires an answer based only on retrieved passages and a clear insufficiency statement when evidence is missing.

In [ ]:
from src.rag.baseline import BaselineRAG, LocalQwenGenerator

generator = LocalQwenGenerator(config)
rag = BaselineRAG(retriever=retriever, generator=generator, config=config)
result = rag.answer_question(sample_question)

print('Question:')
print(result.question)
print('\nAnswer:')
print(result.answer)

## Multiple examples

These questions target different topics so you can verify that retrieval changes with the query.

In [ ]:
example_questions = [
    'Who invented the World Wide Web and where did they work?',
    'How many chambers does the human heart have?',
    'What event happens when the Moon passes between Earth and the Sun?',
]

for question in example_questions:
    print('\n' + '=' * 80)
    display_retrieval(question)
    answer = rag.answer_question(question)
    print('\nGenerated answer:', answer.answer)

### Optional model upgrade

If your Colab GPU has enough free VRAM, change only `generation_model_name` in `RAGConfig` to `Qwen/Qwen2.5-3B-Instruct`, then rerun the index/generation setup cells.